In [1]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import numpy as np

from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from tqdm import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt

from sv3.nn import FunctionalModelJac, MLP, SmallResNet, SmallCNN
from sv3.svd_sgd import SVDOptimizer

import sys
sys.path.append('..')
import copy

from experiments.experiment_code.experiment_utils import train_loop_standard, train_loop_svd

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Using device: {device}")

Using device: cuda


In [2]:
from torchvision import datasets, transforms
from experiments.datasets import CIFAR10Dataset

cifar = CIFAR10Dataset()
train_dataset = cifar.train_dataset
test_dataset = cifar.val_dataset

In [3]:
model_base = SmallResNet(num_classes=10,width=8)
n_params = sum(p.numel() for p in model_base.parameters())
print(f"Number of parameters in the model: {n_params}")
init_state = copy.deepcopy(model_base.state_dict())
del model_base # free memory

Number of parameters in the model: 44370


In [7]:
LOADER_SEED = 645297
batch_size = 64
n_epoch = 5
K = 32
RTOL = 1e-2
LR = 100.0

### train adam

In [ ]:
model_adam = SmallResNet(num_classes=10,width=8)
model_adam.load_state_dict(init_state)
model_adam = model_adam.to(device)

optimizer = torch.optim.Adam(model_adam.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

train_loader = DataLoader(cifar.train_dataset, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(LOADER_SEED))
test_loader = DataLoader(cifar.val_dataset, batch_size=256, shuffle=False)

model_adam, losses_adam = train_loop_standard(model_adam, optimizer, loss_fn, train_loader, test_loader, n_epoch, device, track_acc=True)

Using device cuda


 20%|██        | 1/5 [00:15<01:01, 15.29s/it]

### train svd

In [6]:
model_svd = SmallResNet(num_classes=10, width=8)
model_svd.load_state_dict(init_state)
model_svd = model_svd.to(device)

def loss_fn(pred,y):
    fn = torch.nn.CrossEntropyLoss(reduction='none')
    loss = fn(pred, y).squeeze()
    return loss


model_svd = FunctionalModelJac(model_svd, loss_fn, device=device)
optimizer = SVDOptimizer(model_svd,lr=LR,k=K,rtol=RTOL,track_svd_info=True,svd_mode='randomized',use_rmsprop=True)


train_loader = DataLoader(cifar.train_dataset, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(LOADER_SEED),drop_last=True)
test_loader = DataLoader(cifar.val_dataset, batch_size=256, shuffle=False)

model_svd, losses_svd, optimizer = train_loop_svd(model_svd,optimizer,loss_fn,train_loader,test_loader,n_epoch,device,track_acc=True)
svd_info = optimizer.svd_info

torch.compiler.reset()

  0%|          | 0/5 [00:00<?, ?it/s]/n/home11/sambt/.conda/envs/jax/lib/python3.12/site-packages/torch/_inductor/compile_fx.py:312: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
/n/home11/sambt/.conda/envs/jax/lib/python3.12/site-packages/torch/_inductor/lowering.py:1744: FutureWarning: `torch._prims_common.check` is deprecated and will be removed in the future. Please use `torch._check*` functions instead.
  check(
 40%|████      | 2/5 [03:36<05:24, 108.26s/it]


KeyboardInterrupt: 

### make plots

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(np.arange(len(losses_adam['train'])), losses_adam['train'], label='Adam Train Loss')
ax.plot(np.arange(len(losses_svd['train'])), losses_svd['train'], label=f'SVD Train Loss K={K}')
plt.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(np.arange(len(losses_adam['train_batch'])), losses_adam['train_batch'], label='Adam Train Loss')
ax.plot(np.arange(len(losses_svd['train_batch'])), losses_svd['train_batch'], label=f'SVD Train Loss K={K}')
plt.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
smoothing = 20
ax.plot(np.convolve(losses_adam['train_batch'], np.ones(smoothing)/smoothing, mode='valid'), label='Adam Train Loss')
ax.plot(np.convolve(losses_svd['train_batch'], np.ones(smoothing)/smoothing, mode='valid'), label=f'SVD Train Loss K={K}')
plt.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(np.arange(len(losses_adam['train_batch'])), losses_adam['train_batch'], label='Adam Train Loss')
ax.plot(np.arange(len(losses_svd['train_batch'])), losses_svd['train_batch'], label=f'SVD Train Loss K={K}')
ax.set_xlim(0,500)
plt.legend()

In [ ]:
fig, ax1 = plt.subplots(figsize=(8,8))

plots = []
p1 = ax1.plot(np.arange(len(losses_svd['train_batch'])), losses_svd['train_batch'], 'C0', label=f'SVD Train Loss K={K}')
plots.append(p1[0])
ax1.set_xlabel('Batch')
ax1.set_ylabel('SVD Train Loss')
ax1.legend()

ax2 = ax1.twinx()
p2 = ax2.plot(np.arange(len(svd_info['num_nonzero_svs'])), svd_info['num_nonzero_svs'], 'C1-',label=f'Number of Nonzero SVs K={K}')

ax1.legend(handles=[p1[0],p2[0]])
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.plot(np.arange(len(losses_adam['val'])), losses_adam['val'], label='Adam Validation Loss')
ax.plot(np.arange(len(losses_svd['val'])), losses_svd['val'], label='SVD Validation Loss')
plt.yscale('log')
plt.legend()

In [ ]:
losses_adam['val']

In [ ]:
losses_svd['val']